# LatentMAS-interp — GPU Test Suite

Pulls the repo, installs deps, runs logic tests + end-to-end smoke on real GPU.

**Gates (stop if any fails):**
1. W_a ≠ identity on Qwen3-4B (~2 min)
2. Bucket-1 rate ≥ 8% on 50 GSM8K examples (~15 min)
3. LatentMAS beats Best-of-N by ≥3pp (~6 hr)

Run cells top to bottom. Each gate raises SystemExit if it fails.

In [ ]:
# 1. Clone / update repo
import os
REPO = "https://github.com/spraldev/LatentMAS-interp"
REPO_DIR = "/workspace/LatentMAS-interp"

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO} {REPO_DIR}

os.chdir(REPO_DIR)
print("cwd:", os.getcwd())

In [ ]:
# 2. Install deps
!pip install -q torch transformers accelerate datasets tqdm numpy

In [ ]:
# 3. Confirm GPU is visible
import torch
assert torch.cuda.is_available(), "No GPU found — wrong runtime?"
print(f"GPU: {torch.cuda.get_device_name(0)}  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# 4. Logic unit tests (no model, pure math/IO — should be <30s)
!python test_pipeline.py -v

In [ ]:
# 5. GATE 1: W_a non-trivial on Qwen3-4B (~2 min, $0.02)
# If W_a = identity, Exp A/L/M/Q are all invalid. Stop here.
import os, json
from pathlib import Path
import torch

WA_GATE_DIR = "/workspace/runs/wa_gate"
os.makedirs(WA_GATE_DIR, exist_ok=True)

!python final_run.py \
    --output_dir {WA_GATE_DIR} \
    --model_name Qwen/Qwen3-4B \
    --conditions latent_mas \
    --tasks gsm8k \
    --max_samples 1 \
    --latent_space_realign

wa_path = Path(WA_GATE_DIR) / "wa_matrix.pt"
assert wa_path.exists(), f"wa_matrix.pt not found — run failed"

W = torch.load(wa_path, weights_only=False)["W_a"].float()
S = torch.linalg.svdvals(W)
cond = (S.max() / S.min()).item()
frob = (W - torch.eye(W.shape[0])).norm().item()
frac_near_1 = ((S - 1).abs() < 0.05).float().mean().item()

print(f"W_a shape:           {tuple(W.shape)}")
print(f"cond number:         {cond:.4f}  (want > 1.5)")
print(f"Frob dist from I:    {frob:.4f}  (want > 1.0)")
print(f"frac singvals ≈ 1:   {frac_near_1:.3f}  (want < 0.5)")
print()

if cond > 1.5 and frob > 1.0:
    print("✅ GATE 1 PASSED — W_a non-trivial, proceed")
else:
    print("🛑 GATE 1 FAILED — W_a still identity on Qwen3-4B")
    print("   Pivot to negative result paper. Drop Exp A/L/M/Q.")
    raise SystemExit("W_a gate failed")

In [ ]:
# 6. GATE 2: Bucket-1 rate ≥ 8% on 50 GSM8K examples (~15 min, $0.20)
# If B1 < 5%, Exp B/P/Q have no data — causal experiments are empty.
BUCKET_GATE_DIR = "/workspace/runs/bucket_gate"
os.makedirs(BUCKET_GATE_DIR, exist_ok=True)

!python final_run.py \
    --output_dir {BUCKET_GATE_DIR} \
    --model_name Qwen/Qwen3-4B \
    --conditions latent_mas single_agent_latent_greedy \
    --tasks gsm8k \
    --max_samples 50 \
    --latent_space_realign

from collections import Counter
bucket_file = Path(BUCKET_GATE_DIR) / "buckets" / "gsm8k.json"
assert bucket_file.exists(), "buckets/gsm8k.json missing — did both conditions run?"

rows = json.loads(bucket_file.read_text())
c = Counter(r["bucket"] for r in rows)
n = len(rows)
b1_rate = c[1] / n

print(f"Buckets: B1={c[1]} B2={c[2]} B3={c[3]} B4={c[4]}  (n={n})")
print(f"Bucket-1 rate: {b1_rate*100:.1f}%  (need ≥ 8% for Exp B/P/Q to have data)")
print()

lmas_metas = list(Path(BUCKET_GATE_DIR).glob("latent_mas/gsm8k/example_*/metadata.json"))
sa_metas   = list(Path(BUCKET_GATE_DIR).glob("single_agent_latent_greedy/gsm8k/example_*/metadata.json"))
lmas_acc = sum(json.loads(m.read_text()).get("correct", False) for m in lmas_metas) / max(len(lmas_metas), 1)
sa_acc   = sum(json.loads(m.read_text()).get("correct", False) for m in sa_metas)   / max(len(sa_metas), 1)
print(f"latent_mas acc:              {lmas_acc*100:.1f}%")
print(f"single_agent acc:            {sa_acc*100:.1f}%")
print(f"Gap (LMAS - SA):             {(lmas_acc - sa_acc)*100:+.1f}pp")
print()

if b1_rate >= 0.05:
    print("✅ GATE 2 PASSED — enough Bucket-1 examples, proceed")
else:
    print("🛑 GATE 2 FAILED — Bucket-1 rate too low")
    print("   Exp B/P/Q will have no data. Consider different tasks or model.")
    raise SystemExit("Bucket-1 gate failed")

In [ ]:
# 7. End-to-end smoke — 5 examples, confirms full pipeline at 4B scale (~10 min)
SMOKE_DIR = "/workspace/runs/smoke"
os.makedirs(SMOKE_DIR, exist_ok=True)

!python final_run.py \
    --output_dir {SMOKE_DIR} \
    --model_name Qwen/Qwen3-4B \
    --conditions latent_mas single_agent_latent_greedy \
    --tasks gsm8k \
    --smoke --test \
    --latent_space_realign

smoke = Path(SMOKE_DIR)
for cond in ["latent_mas", "single_agent_latent_greedy"]:
    metas = list((smoke / cond / "gsm8k").glob("example_*/metadata.json"))
    correct = sum(json.loads(m.read_text()).get("correct", False) for m in metas)
    print(f"{cond}: {correct}/{len(metas)} correct")

In [ ]:
# 8. RISK 1: MBPP+ subprocess scoring
MBPP_SMOKE_DIR = "/workspace/runs/smoke_mbpp"
os.makedirs(MBPP_SMOKE_DIR, exist_ok=True)

!python final_run.py \
    --output_dir {MBPP_SMOKE_DIR} \
    --model_name Qwen/Qwen3-4B \
    --conditions latent_mas \
    --tasks mbppplus \
    --smoke --test \
    --latent_space_realign

mbpp_metas = list(Path(MBPP_SMOKE_DIR).glob("latent_mas/mbppplus/example_*/metadata.json"))
print(f"MBPP+ examples completed: {len(mbpp_metas)}  (want >0, no crash = pass)")
for m in sorted(mbpp_metas):
    d = json.loads(m.read_text())
    print(f"  {m.parent.name}: correct={d.get('correct')}")

In [ ]:
# 9. RISK 2: Resume — second pass must skip completed examples
OOM_SMOKE_DIR = "/workspace/runs/smoke_resume"
os.makedirs(OOM_SMOKE_DIR, exist_ok=True)

!python final_run.py \
    --output_dir {OOM_SMOKE_DIR} \
    --model_name Qwen/Qwen3-4B \
    --conditions latent_mas \
    --tasks gsm8k \
    --smoke --test \
    --latent_space_realign

import time
t0 = time.time()
!python final_run.py \
    --output_dir {OOM_SMOKE_DIR} \
    --model_name Qwen/Qwen3-4B \
    --conditions latent_mas \
    --tasks gsm8k \
    --smoke --test \
    --latent_space_realign
elapsed = time.time() - t0
print(f"Second pass: {elapsed:.1f}s  (want <60s)  resume_ok={elapsed < 60}")

In [ ]:
# 10. RISK 3: Storage budget
import shutil

def dir_size_gb(path):
    return sum(f.stat().st_size for f in Path(path).rglob("*") if f.is_file()) / 1e9

smoke_gb = dir_size_gb(SMOKE_DIR)
scale = (200 / 5) * (17 / 2) * 3
projected_gb = smoke_gb * scale
free_gb = shutil.disk_usage("/workspace").free / 1e9
total_gb = shutil.disk_usage("/workspace").total / 1e9

print(f"Smoke disk:        {smoke_gb:.2f} GB")
print(f"Projected full:    {projected_gb:.1f} GB  (200 ex, 17 cond, 3 tasks)")
print(f"Volume free/total: {free_gb:.0f} / {total_gb:.0f} GB")
if projected_gb > free_gb * 0.8:
    print("⚠️  WARNING: projected >80% of free space — add --no_layer_hidden or --no_kv_latent")
else:
    print("✅ Storage OK")

In [ ]:
# 11. RISK 4: Untested conditions
COND_SMOKE_DIR = "/workspace/runs/smoke_conds"
os.makedirs(COND_SMOKE_DIR, exist_ok=True)

!python final_run.py \
    --output_dir {COND_SMOKE_DIR} \
    --model_name Qwen/Qwen3-4B \
    --conditions topk_gated random_gated exp_m_identity_wa \
    --tasks gsm8k \
    --smoke --test \
    --latent_space_realign

print("\n--- Untested condition results ---")
for cond in ["topk_gated", "random_gated", "exp_m_identity_wa"]:
    metas = list(Path(COND_SMOKE_DIR).glob(f"{cond}/gsm8k/example_*/metadata.json"))
    if not metas:
        print(f"  {cond}: NO OUTPUT — crashed or condition name wrong")
        continue
    correct = sum(json.loads(m.read_text()).get("correct", False) for m in metas)
    print(f"  {cond}: {len(metas)} examples, {correct}/{len(metas)} correct")

In [ ]:
# 12. GATE 3: LatentMAS beats Best-of-N by ≥3pp (~6 hr, $4)
# FATAL 3: if LMAS doesn't clear this bar, the headline claim is dead.
# Run AFTER gates 1 and 2 pass. Uses 200 examples on GSM8K.
BON_GATE_DIR = "/workspace/runs/bon_gate"
os.makedirs(BON_GATE_DIR, exist_ok=True)

!python final_run.py \
    --output_dir {BON_GATE_DIR} \
    --model_name Qwen/Qwen3-4B \
    --conditions latent_mas best_of_n \
    --tasks gsm8k \
    --max_samples 200 \
    --latent_space_realign

def acc(cond, task, base):
    metas = list(Path(base).glob(f"{cond}/{task}/example_*/metadata.json"))
    if not metas:
        return None, 0
    correct = sum(json.loads(m.read_text()).get("correct", False) for m in metas)
    return correct / len(metas), len(metas)

lmas_acc, lmas_n = acc("latent_mas", "gsm8k", BON_GATE_DIR)
bon_acc,  bon_n  = acc("best_of_n",  "gsm8k", BON_GATE_DIR)

print(f"latent_mas:  {lmas_acc*100:.1f}%  (n={lmas_n})")
print(f"best_of_n:   {bon_acc*100:.1f}%  (n={bon_n})")
gap = (lmas_acc - bon_acc) * 100
print(f"Gap:         {gap:+.1f}pp  (need ≥ +3pp to pass FATAL 3)")
print()

if gap >= 3.0:
    print("✅ GATE 3 PASSED — LMAS beats Best-of-N, headline claim holds")
    print("   Proceed with full run: all 3 tasks, all 17 conditions")
elif gap >= 0:
    print("⚠️  GATE 3 MARGINAL — LMAS ahead but < 3pp")
    print("   Run arc_challenge + mbppplus before deciding — need ≥2/3 tasks at +3pp")
else:
    print("🛑 GATE 3 FAILED — Best-of-N beats LMAS")
    print("   Headline claim is dead. Reframe paper around mechanistic findings only.")

In [ ]:
# 13. All gates passed — full run commands
FULL_RUN_DIR = "/workspace/runs/full"
print("All gates passed. Full run commands:")
print()
print("# Step 1: GSM8K + ARC (~7 hr)")
print(f"python final_run.py \\")
print(f"    --output_dir {FULL_RUN_DIR} \\")
print(f"    --model_name Qwen/Qwen3-4B \\")
print(f"    --tasks gsm8k arc_challenge \\")
print(f"    --latent_space_realign")
print()
print("# Step 2: MBPP+ separately (~5 hr) — isolates subprocess risk")
print(f"python final_run.py \\")
print(f"    --output_dir {FULL_RUN_DIR} \\")
print(f"    --model_name Qwen/Qwen3-4B \\")
print(f"    --tasks mbppplus \\")
print(f"    --latent_space_realign")